### hybrid_llama.py
#### Hybrid inference
1) ML baseline
2) DL baseline
3) ML + AI
4) DL + AI

In [14]:
import os
import json
import joblib
import numpy as np
import pandas as pd

from openai import OpenAI
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

### Files

In [15]:
SHARED_DATASET_FILE = "../data/processed/cleaned_emotions_shared.csv"
SHARED_SPLIT_FILE = "../data/splits/shared_split.json"

ML_VECTORIZER_FILE = "../models/ml_tfidf_vectorizer.joblib"
LR_MODEL_FILE = "../models/ml_models/logistic_regression.joblib"
SVM_MODEL_FILE = "../models/ml_models/svm.joblib"
NB_MODEL_FILE = "../models/ml_models/naive_bayes.joblib"
RF_MODEL_FILE = "../models/ml_models/random_forest.joblib"

DL_MODEL_FILE = "../models/bilstm_model.keras"
WORD_INDEX_FILE = "../models/word_index.json"
LABEL_MAPPING_FILE = "../models/label_mapping.json"
DL_METADATA_FILE = "../models/dl_metadata.json"

OUTPUT_FILE = "../data/processed/hybrid_llama_results.csv"
SUMMARY_FILE = "../data/processed/hybrid_llama_summary.csv"

os.makedirs("../data/processed", exist_ok=True)

### Load test set

In [16]:
df = pd.read_csv(SHARED_DATASET_FILE)

with open(SHARED_SPLIT_FILE, "r", encoding="utf-8") as f:
    split_dict = json.load(f)

test_df = df[df["row_id"].isin(split_dict["test_ids"])].copy().reset_index(drop=True)

print("Test shape:", test_df.shape)
print(test_df["label"].value_counts())

valid_labels = sorted(df["label"].unique().tolist())

Test shape: (324, 3)
label
neutral    75
disgust    64
fear       60
angry      49
sad        41
happy      35
Name: count, dtype: int64


### Load ML vectorizer

In [17]:
ml_vectorizer = joblib.load(ML_VECTORIZER_FILE)
X_test_ml = ml_vectorizer.transform(test_df["text"]).toarray()


### Load all 4 ML models

In [18]:
lr_model = joblib.load(LR_MODEL_FILE)
svm_model = joblib.load(SVM_MODEL_FILE)
nb_model = joblib.load(NB_MODEL_FILE)
rf_model = joblib.load(RF_MODEL_FILE)

# Predict ML
lr_pred = lr_model.predict(X_test_ml)
lr_proba = lr_model.predict_proba(X_test_ml)
lr_conf = lr_proba.max(axis=1)

svm_pred = svm_model.predict(X_test_ml)
svm_proba = svm_model.predict_proba(X_test_ml)
svm_conf = svm_proba.max(axis=1)

nb_pred = nb_model.predict(X_test_ml)
nb_proba = nb_model.predict_proba(X_test_ml)
nb_conf = nb_proba.max(axis=1)

rf_pred = rf_model.predict(X_test_ml)
rf_proba = rf_model.predict_proba(X_test_ml)
rf_conf = rf_proba.max(axis=1)

### DL baseline

In [19]:
dl_model = load_model(DL_MODEL_FILE)

with open(WORD_INDEX_FILE, "r", encoding="utf-8") as f:
    word_index = json.load(f)

with open(LABEL_MAPPING_FILE, "r", encoding="utf-8") as f:
    label_mapping = json.load(f)

with open(DL_METADATA_FILE, "r", encoding="utf-8") as f:
    dl_metadata = json.load(f)

tokenizer = Tokenizer(
    num_words=dl_metadata["max_vocab_size"],
    oov_token="<OOV>"
)
tokenizer.word_index = word_index

max_sequence_length = dl_metadata["max_sequence_length"]

X_test_seq = tokenizer.texts_to_sequences(test_df["text"])
X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=max_sequence_length,
    padding="post",
    truncating="post"
)

index_to_label = {v: k for k, v in label_mapping.items()}

dl_proba = dl_model.predict(X_test_pad, verbose=0)
dl_pred_idx = np.argmax(dl_proba, axis=1)
dl_pred = [index_to_label[int(i)] for i in dl_pred_idx]
dl_conf = dl_proba.max(axis=1)


### AI setup

In [20]:
client = OpenAI(
    api_key="ollama",
    base_url="http://localhost:11434/v1"
)

LLMAMA_MODEL = "llama3.1:8b"
print("AI ready:", LLMAMA_MODEL)


AI ready: llama3.1:8b


### Helper functions

In [21]:
def normalize_label(answer):
    answer = str(answer).strip().lower()

    for label in valid_labels:
        if answer == label.lower():
            return label

    for label in valid_labels:
        if label.lower() in answer:
            return label

    return ""

def ai_refine(text, baseline_pred, confidence, threshold=0.75):
    # ถ้า model มั่นใจอยู่แล้ว ไม่ต้องเรียก AI
    if confidence >= threshold:
        return baseline_pred

    label_text = ", ".join(valid_labels)

    system_prompt = (
        "You are an emotion classification assistant. "
        "Choose exactly one label from the allowed labels only. "
        "Return only the label."
    )

    user_prompt = f"""
Allowed labels:
{label_text}

Patient utterance:
{text}

Baseline predicted label:
{baseline_pred}

Return exactly one label only.
""".strip()

    try:
        response = client.chat.completions.create(
            model=LLMAMA_MODEL,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0
        )

        answer = response.choices[0].message.content.strip()
        final_label = normalize_label(answer)

        if final_label == "":
            return baseline_pred

        return final_label

    except Exception as e:
        print("AI error:", e)
        return baseline_pred

def evaluate_model(y_true, y_pred, model_name):
    acc = accuracy_score(y_true, y_pred)
    f1_macro = f1_score(y_true, y_pred, average="macro")
    f1_weighted = f1_score(y_true, y_pred, average="weighted")

    print(f"\n===== {model_name} =====")
    print(f"Accuracy     : {acc:.4f}")
    print(f"F1 Macro     : {f1_macro:.4f}")
    print(f"F1 Weighted  : {f1_weighted:.4f}")

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

    return {
        "Model": model_name,
        "Accuracy": acc,
        "F1_Macro": f1_macro,
        "F1_Weighted": f1_weighted
    }

### Save baseline results in dataframe

In [22]:
results_df = test_df[["row_id", "text", "label"]].copy()

results_df["lr_pred"] = lr_pred
results_df["lr_conf"] = lr_conf

results_df["svm_pred"] = svm_pred
results_df["svm_conf"] = svm_conf

results_df["nb_pred"] = nb_pred
results_df["nb_conf"] = nb_conf

results_df["rf_pred"] = rf_pred
results_df["rf_conf"] = rf_conf

results_df["dl_pred"] = dl_pred
results_df["dl_conf"] = dl_conf

### AI refine

In [23]:
lr_ai_pred = []
svm_ai_pred = []
nb_ai_pred = []
rf_ai_pred = []
dl_ai_pred = []

for _, row in results_df.iterrows():
    lr_ai_pred.append(ai_refine(row["text"], row["lr_pred"], row["lr_conf"]))
    svm_ai_pred.append(ai_refine(row["text"], row["svm_pred"], row["svm_conf"]))
    nb_ai_pred.append(ai_refine(row["text"], row["nb_pred"], row["nb_conf"]))
    rf_ai_pred.append(ai_refine(row["text"], row["rf_pred"], row["rf_conf"]))
    dl_ai_pred.append(ai_refine(row["text"], row["dl_pred"], row["dl_conf"]))

results_df["lr_ai_pred"] = lr_ai_pred
results_df["svm_ai_pred"] = svm_ai_pred
results_df["nb_ai_pred"] = nb_ai_pred
results_df["rf_ai_pred"] = rf_ai_pred
results_df["dl_ai_pred"] = dl_ai_pred


### AI refine

In [24]:
lr_ai_pred = []
svm_ai_pred = []
nb_ai_pred = []
rf_ai_pred = []
dl_ai_pred = []

for _, row in results_df.iterrows():
    lr_ai_pred.append(ai_refine(row["text"], row["lr_pred"], row["lr_conf"]))
    svm_ai_pred.append(ai_refine(row["text"], row["svm_pred"], row["svm_conf"]))
    nb_ai_pred.append(ai_refine(row["text"], row["nb_pred"], row["nb_conf"]))
    rf_ai_pred.append(ai_refine(row["text"], row["rf_pred"], row["rf_conf"]))
    dl_ai_pred.append(ai_refine(row["text"], row["dl_pred"], row["dl_conf"]))

results_df["lr_ai_pred"] = lr_ai_pred
results_df["svm_ai_pred"] = svm_ai_pred
results_df["nb_ai_pred"] = nb_ai_pred
results_df["rf_ai_pred"] = rf_ai_pred
results_df["dl_ai_pred"] = dl_ai_pred

## Evaluate all

In [25]:
summary_rows = []

summary_rows.append(evaluate_model(results_df["label"], results_df["lr_pred"], "Logistic Regression"))
summary_rows.append(evaluate_model(results_df["label"], results_df["svm_pred"], "Support Vector Machine"))
summary_rows.append(evaluate_model(results_df["label"], results_df["nb_pred"], "Naive Bayes"))
summary_rows.append(evaluate_model(results_df["label"], results_df["rf_pred"], "Random Forest"))
summary_rows.append(evaluate_model(results_df["label"], results_df["dl_pred"], "DL"))

summary_rows.append(evaluate_model(results_df["label"], results_df["lr_ai_pred"], "Logistic Regression + AI"))
summary_rows.append(evaluate_model(results_df["label"], results_df["svm_ai_pred"], "Support Vector Machine + AI"))
summary_rows.append(evaluate_model(results_df["label"], results_df["nb_ai_pred"], "Naive Bayes + AI"))
summary_rows.append(evaluate_model(results_df["label"], results_df["rf_ai_pred"], "Random Forest + AI"))
summary_rows.append(evaluate_model(results_df["label"], results_df["dl_ai_pred"], "DL + AI"))

summary_df = pd.DataFrame(summary_rows).sort_values(by="F1_Macro", ascending=False)

print("\n===== Summary =====")
print(summary_df)


===== Logistic Regression =====
Accuracy     : 0.9074
F1 Macro     : 0.9229
F1 Weighted  : 0.9075

Classification Report:
              precision    recall  f1-score   support

       angry       1.00      1.00      1.00        49
     disgust       0.87      0.92      0.89        64
        fear       0.84      0.82      0.83        60
       happy       1.00      0.97      0.99        35
     neutral       0.84      0.84      0.84        75
         sad       1.00      0.98      0.99        41

    accuracy                           0.91       324
   macro avg       0.93      0.92      0.92       324
weighted avg       0.91      0.91      0.91       324


Confusion Matrix:
[[49  0  0  0  0  0]
 [ 0 59  1  0  4  0]
 [ 0  4 49  0  7  0]
 [ 0  0  0 34  1  0]
 [ 0  5  7  0 63  0]
 [ 0  0  1  0  0 40]]

===== Support Vector Machine =====
Accuracy     : 0.9228
F1 Macro     : 0.9355
F1 Weighted  : 0.9229

Classification Report:
              precision    recall  f1-score   support

       

### Save files

In [26]:
results_df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
summary_df.to_csv(SUMMARY_FILE, index=False, encoding="utf-8-sig")

print("\nSaved files:")
print("-", OUTPUT_FILE)
print("-", SUMMARY_FILE)


Saved files:
- ../data/processed/hybrid_llama_results.csv
- ../data/processed/hybrid_llama_summary.csv
